# Assignment 07: From-Scratch Normalization & Encoding (100 points)

## Context

Before data reaches a model, it must be **normalized** (so features are on comparable scales) and **encoded** (so categorical data becomes numeric). In USAAIO, you implement these from scratch -- no sklearn.

The critical concept is the **fit/transform** pattern: compute statistics (mean, std, category mappings) from the **training set only**, then apply them to both training and test sets. Using test-set statistics during fitting is called **data leakage** and invalidates your results.

### Key Techniques

- **Z-score standardization**: $z = (x - \mu) / \sigma$ -- centers at 0, unit variance
- **Min-max normalization**: $x_{\text{norm}} = (x - x_{\min}) / (x_{\max} - x_{\min})$ -- scales to [0, 1]
- **One-hot encoding**: integer label $k$ becomes a vector with 1 at position $k$, 0 elsewhere
- **Label encoding**: string categories mapped to consecutive integers

### Notation

- Shape annotations are required: `# (N, D) - (D,) -> (N, D)` via broadcasting
- `N` = number of samples, `D` = number of features, `C` = number of classes

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import pandas as pd
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else** for the following purposes:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.
>
> - **No sklearn, no explicit `for`/`while` loops** unless explicitly permitted.
> - Implement everything using NumPy/Pandas vectorized operations.
> - Every function must work on arbitrary-shaped inputs.

---

## Part 1 (20 points, coding task)

**Reasoning is not required.**

Implement Z-score standardization from scratch using the fit/transform pattern.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
# Training and test data with different scales per feature
X_train = np.random.randn(80, 4) * np.array([100, 1, 0.01, 50]) + np.array([500, 0, 5, -100])
X_test = np.random.randn(20, 4) * np.array([100, 1, 0.01, 50]) + np.array([500, 0, 5, -100])
print("X_train shape:", X_train.shape)  # (80, 4)
print("X_test shape:", X_test.shape)    # (20, 4)
print("Column means:", X_train.mean(axis=0).round(2))
print("Column stds:", X_train.std(axis=0).round(2))

1. **(7 pts)** Implement `z_score_fit(X_train)` that computes and returns the per-column mean and standard deviation from training data. Each should have shape `(D,)`.

2. **(7 pts)** Implement `z_score_transform(X, mean, std)` that applies $z = (X - \mu) / \sigma$ using broadcasting. Handle the edge case where `std == 0` by leaving that column unchanged (divide by 1 instead).

3. **(6 pts)** Fit on `X_train`, then transform **both** `X_train` and `X_test` using the **training** statistics. Store in `X_train_z` and `X_test_z`. Verify that training columns have mean $\approx 0$ and std $\approx 1$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def z_score_fit(X_train):
    """Compute mean and std from training data.
    Returns: (mean, std) each of shape (D,)
    """
    pass

def z_score_transform(X, mean, std):
    """Apply z-score normalization.
    X shape: (N, D), mean shape: (D,), std shape: (D,)
    Returns: normalized X of shape (N, D)
    """
    pass

mean, std = z_score_fit(X_train)
X_train_z = z_score_transform(X_train, mean, std)
X_test_z = z_score_transform(X_test, mean, std)

""" END OF THIS PART """

---

Min-max normalization scales features to a fixed range, typically [0, 1]. Unlike Z-score, it guarantees bounded outputs -- which matters when your model expects inputs in a specific range (e.g., pixel values). The tradeoff: it is more sensitive to outliers since a single extreme value stretches the entire range.

---

## Part 2 (20 points, coding task)

**Reasoning is not required.**

Implement min-max normalization from scratch.

1. **(7 pts)** Implement `minmax_fit(X_train)` that computes and returns the per-column min and max from training data. Each should have shape `(D,)`.

2. **(7 pts)** Implement `minmax_transform(X, min_vals, max_vals)` that applies $x_{\text{norm}} = (x - x_{\min}) / (x_{\max} - x_{\min})$. Handle the edge case where `max == min` by leaving that column as 0.

3. **(6 pts)** Fit on `X_train`, then transform both sets. Store in `X_train_mm` and `X_test_mm`. Verify that training columns have min $= 0$ and max $= 1$. Note: test data may slightly exceed [0, 1] -- explain why this is expected.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def minmax_fit(X_train):
    """Compute min and max from training data.
    Returns: (min_vals, max_vals) each of shape (D,)
    """
    pass

def minmax_transform(X, min_vals, max_vals):
    """Apply min-max normalization.
    Returns: normalized X with values in [0, 1] for training data.
    """
    pass

min_vals, max_vals = minmax_fit(X_train)
X_train_mm = minmax_transform(X_train, min_vals, max_vals)
X_test_mm = minmax_transform(X_test, min_vals, max_vals)

""" END OF THIS PART """

---

Parts 1 and 2 handle numeric features. But real datasets contain **categorical features** -- strings like `"red"`, `"blue"`, `"green"` that have no numeric ordering. Models need numbers, so we must encode these categories. The two standard approaches are one-hot encoding (for nominal data) and label encoding (for ordinal data or tree-based models).

---

## Part 3 (20 points, coding task)

**Reasoning is not required.**

Implement one-hot encoding and label encoding from scratch.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
int_labels = np.array([0, 2, 1, 0, 3, 2, 1])  # (7,)
num_classes = 4

train_colors = np.array(['red', 'blue', 'green', 'red', 'blue', 'green', 'red'])
test_colors = np.array(['blue', 'red', 'yellow', 'green'])  # 'yellow' is unseen!

1. **(7 pts)** Implement `one_hot_encode(labels, num_classes)` that converts integer labels of shape `(N,)` to a binary matrix of shape `(N, num_classes)`. **No loops.** Use either broadcasting (`labels[:, None] == np.arange(num_classes)`) or fancy indexing. Verify that `np.argmax` recovers the original labels.

2. **(7 pts)** Implement `label_encode_fit(categories)` that learns a mapping from unique string categories to consecutive integers (sorted alphabetically). Returns a dictionary. Then implement `label_encode_transform(categories, mapping)` that applies the mapping. **Unknown categories** (not seen in training) should map to `-1`.

3. **(6 pts)** Apply label encoding: fit on `train_colors`, transform both `train_colors` and `test_colors`. Store in `train_encoded` and `test_encoded`. Verify that `'yellow'` maps to `-1`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def one_hot_encode(labels, num_classes=None):
    """Convert integer labels to one-hot vectors.
    labels shape: (N,) of integers in [0, num_classes)
    Returns: (N, num_classes) float array
    """
    pass

def label_encode_fit(categories):
    """Learn mapping from category strings to integers (sorted alphabetically).
    Returns: dict mapping string -> int
    """
    pass

def label_encode_transform(categories, mapping):
    """Apply the learned mapping. Unknown categories map to -1.
    Returns: numpy array of integers
    """
    pass

# One-hot
encoded_oh = one_hot_encode(int_labels, num_classes)

# Label encoding
mapping = label_encode_fit(train_colors)
train_encoded = label_encode_transform(train_colors, mapping)
test_encoded = label_encode_transform(test_colors, mapping)

""" END OF THIS PART """

---

## Part 4 (15 points, non-coding task)

**Reasoning is required.**

Answer the following questions about normalization and encoding.

1. **(5 pts)** Why must you fit normalization statistics on the **training set only** and not on the entire dataset? Give a concrete example of how using full-dataset statistics could lead to misleadingly good test performance.

2. **(5 pts)** When would you choose min-max normalization over Z-score standardization, and vice versa? Give one scenario where each is clearly better.

3. **(5 pts)** One-hot encoding a feature with 1000 unique categories creates 1000 columns. Why is this problematic? Name two alternative encoding strategies that reduce dimensionality.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

The final test: combine everything into a complete preprocessing pipeline. Real datasets have mixed types -- numeric columns that need normalization and categorical columns that need encoding. The pipeline must handle all of them while maintaining the fit/transform discipline.

---

## Part 5 (25 points, coding task)

**Reasoning is not required.**

Build a complete preprocessing pipeline for a mixed-type dataset.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
df = pd.DataFrame({
    'age': [25, 30, 22, 45, 35, 28, 50, 33, 27, 40],
    'income': [50000, 75000, 35000, 120000, 85000, 60000, 130000, 70000, 45000, 95000],
    'education': ['bachelors', 'masters', 'high_school', 'phd', 'masters',
                  'bachelors', 'phd', 'masters', 'bachelors', 'masters'],
    'city': ['NYC', 'LA', 'NYC', 'SF', 'LA', 'NYC', 'SF', 'LA', 'NYC', 'SF'],
    'score': [72, 85, 60, 95, 88, 76, 92, 82, 68, 90]
})

# Split into train (first 8) and test (last 2)
df_train = df.iloc[:8].copy()
df_test = df.iloc[8:].copy()
print(df_train)
print(df_test)

Build a preprocessing pipeline that:

1. **(5 pts)** Z-score normalizes the numeric columns `'age'` and `'income'` (fit on train, transform both).

2. **(5 pts)** One-hot encodes `'education'` and `'city'` (fit categories on train, handle unknown categories in test by assigning all-zero vectors).

3. **(5 pts)** Separates `'score'` as the target variable (do not normalize it).

4. **(5 pts)** Concatenates all processed features into single NumPy arrays: `X_train_final`, `X_test_final`, `y_train_final`, `y_test_final`.

5. **(5 pts)** Verifies the pipeline: print the final shapes, confirm numeric columns have mean $\approx 0$ in training data, and confirm one-hot columns sum correctly.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

X_train_final = None  # shape: (8, num_features)
y_train_final = None  # shape: (8,)
X_test_final = None   # shape: (2, num_features)
y_test_final = None   # shape: (2,)

""" END OF THIS PART """